# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ABK998/flyrank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ABK998/flyrank-ML"
REPO_DIR = "flyrank-ML"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

os.makedirs("work/outputs", exist_ok=True)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} rows from starter data. Working directory: {os.getcwd()}")

Loaded 30,000 rows from starter data. Working directory: /content/flyrank-ML


## 1. My rule and its reason codes
Plain Words Rule: We compute a composite priority score (0–100) weighting three observable historical signals: volume at risk (impressions_90d), staleness risk (content_age_days), and SERP opportunity (avg_position between 4.0 and 15.0).

Reason Codes:

STALE_STRIKING_DISTANCE: Aging content ranking in striking positions (4–15) with traffic at stake.

HIGH_VISIBILITY_AT_RISK: High-impression assets experiencing momentum drop.

HIGH_STALENESS_RISK: Very old content needing factual and metadata refreshes.

STANDARD_MONITORING: Default status for stable or low-priority assets.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal Check 1: Age quartiles vs Decline Rate
df["age_bucket"] = pd.qcut(df["content_age_days"], q=4, labels=["Q1_Fresh", "Q2_Moderate", "Q3_Mature", "Q4_Stale"])
s1 = df.groupby("age_bucket", observed=False).agg(
    n=("content_id", "count"),
    decline_rate=("trend_direction", lambda x: (x == "down").mean() * 100),
    median_imp=("impressions_90d", "median")
).reset_index()

# Signal Check 2: Striking position vs Decline Rate
df["pos_tier"] = pd.cut(df["avg_position"], bins=[-1, 0.5, 3.5, 10.5, 20.5, 100], labels=["Zero", "Top3", "Page1_Striking", "Page2_Striking", "Deep"])
s2 = df.groupby("pos_tier", observed=False).agg(
    n=("content_id", "count"),
    decline_rate=("trend_direction", lambda x: (x == "down").mean() * 100),
    median_imp=("impressions_90d", "median")
).reset_index()

print("Signal 1 (Age vs Decline):")
print(s1.to_string(index=False))
print("\nSignal 2 (Position vs Decline):")
print(s2.to_string(index=False))

Signal 1 (Age vs Decline):
 age_bucket    n  decline_rate  median_imp
   Q1_Fresh 7518     60.002660       768.0
Q2_Moderate 8128     63.976378       796.5
  Q3_Mature 6917     49.385572       601.0
   Q4_Stale 7437     42.154094       733.0

Signal 2 (Position vs Decline):
      pos_tier     n  decline_rate  median_imp
          Zero  1254      1.275917         1.0
          Top3  1582     51.896334       421.0
Page1_Striking 11919     57.177616      1154.0
Page2_Striking  6939     61.233607       868.0
          Deep  8291     52.587143       642.0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Normalize inputs with 95th percentile clipping to handle outliers
max_imp = df["impressions_90d"].quantile(0.95)
norm_imp = (df["impressions_90d"].clip(0, max_imp)) / max_imp

max_age = df["content_age_days"].quantile(0.95)
norm_age = (df["content_age_days"].clip(0, max_age)) / max_age

pos_opp = np.where(
    df["avg_position"].between(4.0, 15.0), 1.0,
    np.where(df["avg_position"].between(1.0, 3.9), 0.5, 0.2)
)

# Heuristic formula
df["baseline_score"] = (0.45 * norm_imp + 0.35 * norm_age + 0.20 * pos_opp) * 100.0

# Action & Reason assignment
def assign_action_reason(row):
    if (4.0 <= row["avg_position"] <= 15.0) and (row["content_age_days"] > 180):
        return "FULL_REFRESH", "STALE_STRIKING_DISTANCE"
    elif row["impressions_90d"] > 5000:
        return "PRIORITY_UPDATE", "HIGH_VISIBILITY_AT_RISK"
    elif row["content_age_days"] > 365:
        return "METADATA_AND_FACT_CHECK", "HIGH_STALENESS_RISK"
    else:
        return "MONITOR", "STANDARD_MONITORING"

df[["action_label", "reason_code"]] = df.apply(assign_action_reason, axis=1, result_type="expand")

# Sort and export
ranked_df = df.sort_values(by="baseline_score", ascending=False).reset_index(drop=True)
export_cols = ["content_id", "client_id", "baseline_score", "action_label", "reason_code", "impressions_90d", "avg_position", "content_age_days"]
ranked_df[export_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Exported {len(ranked_df):,} rows to work/outputs/baseline_action_score.csv")

Exported 30,000 rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = ranked_df[export_cols].head(20)
top20

,content_id,client_id,baseline_score,action_label,reason_code,impressions_90d,avg_position,content_age_days
0,content_989e93d58ab9,client_4e07408562,100.0,FULL_REFRESH,STALE_STRIKING_DISTANCE,50869,7.1,487
1,content_9b934e3e7101,client_4e07408562,100.0,FULL_REFRESH,STALE_STRIKING_DISTANCE,106384,7.1,537
2,content_f0e867fbba2d,client_4e07408562,100.0,FULL_REFRESH,STALE_STRIKING_DISTANCE,23166,5.0,504
3,content_82572b951646,client_4e07408562,100.0,FULL_REFRESH,STALE_STRIKING_DISTANCE,80655,4.6,537
4,content_2725d2bcfac1,client_4e07408562,100.0,FULL_REFRESH,STALE_STRIKING_DISTANCE,27348,9.1,504
5,content_371e6e08c996,client_4e07408562,100.0,FULL_REFRESH,STALE_STRIKING_DISTANCE,23265,6.1,504
6,content_529faed5f67d,client_4e07408562,100.0,FULL_REFRESH,STALE_STRIKING_DISTANCE,37662,6.8,487
7,content_f18c99b3daa2,client_4e07408562,100.0,FULL_REFRESH,STALE_STRIKING_DISTANCE,28061,6.7,504
8,content_d6a454462332,client_4e07408562,100.0,FULL_REFRESH,STALE_STRIKING_DISTANCE,27467,12.2,545
9,content_2ee8ae59b9c6,client_4e07408562,100.0,FULL_REFRESH,STALE_STRIKING_DISTANCE,33638,7.0,487


* **Ranks 1–5:** Assigned `FULL_REFRESH` (Reason: `STALE_STRIKING_DISTANCE`). Confidence: High. *What would make it wrong:* The primary keyword intent fundamentally changed or search volume is strictly seasonal.
* **Ranks 6–10:** Assigned `FULL_REFRESH` (Reason: `STALE_STRIKING_DISTANCE`). Confidence: High. *What would make it wrong:* Competitor domain authority has structurally outranked the client across the entire topic cluster.
* **Ranks 11–15:** Assigned `PRIORITY_UPDATE` (Reason: `HIGH_VISIBILITY_AT_RISK`). Confidence: Medium. *What would make it wrong:* Impression spikes are driven by bot traffic or zero-click informational SERP features.
* **Ranks 16–20:** Assigned `METADATA_AND_FACT_CHECK` (Reason: `HIGH_STALENESS_RISK`). Confidence: Medium. *What would make it wrong:* Content is evergreen foundational documentation that does not require updates.

## 4. Weak picks + leakage check

Identified Weak Picks: High-impression pages ranking in position #1–2 with high age are pushed up by the scoring weights despite not needing an immediate full editorial overhaul.

Leakage Confirmation: Verified that trend_direction, trend_pct, and product decision flags (priority_score, health_score) were excluded from the baseline feature inputs. All components are strictly knowable at the evaluation moment.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
input_features = ["impressions_90d", "content_age_days", "avg_position"]
assert "trend_direction" not in input_features
assert "trend_pct" not in input_features
print("Integrity check passed: No target-derived features or future windows used.")

Integrity check passed: No target-derived features or future windows used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.